# Search and assess agents in the Global A2A Registry

This notebook has one workflow: **search for agents → compare search results → inspect one agent → review its verification evidence**. It uses read-only public endpoints and retains raw responses for inspection.

## How to interpret the scores

Keep these signals separate:

- **Relevance score** is returned by search. It measures match quality, not trust.
- **Verification rank** is the registry's ordinal identity level: `unclaimed/unverified = 0`, `github_verified = 1`, `domain_verified = 2`, and `ans_verified = 3`. It is not a percentage.
- **Readiness score** is the registry validator's 0–100 assessment of the Agent Card's schema, network, discovery, and trust-artifact readiness. Read its findings alongside the number.
- **Trust score** is intentionally not calculated. Registry identity and card readiness do not establish behavioral reliability, authorization safety, or capability accuracy.

In [3]:
import importlib
import json
import sys
from pathlib import Path

# Make imports work whether Jupyter starts in the repository root or in EXPERIMENTS.
search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (path for path in search_roots if (path / "EXPERIMENTS" / "utils.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate EXPERIMENTS/utils.py from the current directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import EXPERIMENTS.utils as experiment_utils

# Reload edits made to utils.py without requiring a kernel restart.
experiment_utils = importlib.reload(experiment_utils)
assess_agent = experiment_utils.a2a_registry_assess_agent
print_table = experiment_utils.a2a_registry_print_table
search_agents = experiment_utils.a2a_registry_search_agents

QUERY = "cafe"
VALIDATE_TOP = 3  # Each validation makes one network request.

SEARCH_FILTERS = {
    # "category": "Utilities",
    # "target": "General",
    # "sort": "newest",
    # "payment_model": "free",
}

## 1. Search for agents

Edit `QUERY` and optional filters above, then run this cell. The full HTTP response remains available in `search_result["response"]`.

In [4]:
search_result = search_agents(QUERY, **SEARCH_FILTERS)
agents = search_result["agents"]
response = search_result["response"]

print(f"HTTP {response['status']} | {response['elapsedMs']} ms | {response['url']}")
print(f"Registry reported {search_result['total']} result(s); this page contains {len(agents)}.")
print_table(
    agents,
    ["package", "name", "category", "protocol", "relevanceScore",
     "verificationLevel", "verificationRank", "manifest"],
    limit=50,
)

HTTP 200 | 1100.2 ms | https://api.a2a-registry.org/public/agents?q=cafe&page=1
Registry reported 2 result(s); this page contains 2.
package                                            | name                                          | category  | protocol | relevanceScore | verificationLevel | verificationRank | manifest                                          
---------------------------------------------------+-----------------------------------------------+-----------+----------+----------------+-------------------+------------------+---------------------------------------------------
dev.workers.digital_agent_cafe                     | Digital Agent Cafe                            | Utilities | a2a      | 0.56           | unclaimed         | 0                | https://dac-public-free-discovery.donuk-kitap.wor…
com.digitalagentcafe.where_digital_entities_meet_… | Where Digital Entities Meet — Discovery Guide | Utilities | a2a      | 0.51           | unclaimed         | 0            

### All search-result details

This prints every field returned by the registry for every agent on the current results page. The `_derivedDomains` object is calculated locally from the returned manifest and OpenAPI URLs; it is not a registry-provided claim. The unmodified response remains in `response["data"]`.

**Insights** 🍀
  * Use `verificationLevel` tells more info than `isVerified`, it should cover value:
    * **GitHub verification**: the developer connects GitHub, proves ownership of the relevant repository/account, and claims or publishes the agent.
    * **Domain verification**: the organization proves control of its domain with an a2a-registry-verify=<token> DNS TXT record.
    * **ANS verification**: the domain is verified through GoDaddy Agent Name Service using _agent.<domain> records and its certificate-backed mechanism. 
    * **UNCLAIMED**: An anonymously submitted agent.

In [5]:
from urllib.parse import urlparse

all_search_details = []
for agent in agents:
    details = dict(agent["rawRegistryRecord"])
    manifest_url = details.get("manifestUrl") or details.get("manifest_url")
    openapi_url = details.get("openapiUrl") or details.get("openapi_url")
    details["_derivedDomains"] = {
        "manifestDomain": urlparse(manifest_url).hostname if manifest_url else None,
        "openapiDomain": urlparse(openapi_url).hostname if openapi_url else None,
    }
    all_search_details.append(details)

print(json.dumps(all_search_details, indent=2, ensure_ascii=False))

[
  {
    "id": "0c1c5117-e8c9-4c3f-bb48-d35a58d7b3dd",
    "orgId": "org_system_unclaimed",
    "packageName": "dev.workers.digital_agent_cafe",
    "displayName": "Digital Agent Cafe",
    "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled digital persons. CHAT is available. No monetary payment is required.",
    "targetAudience": "General",
    "category": "Utilities",
    "manifestUrl": "https://dac-public-free-discovery.donuk-kitap.workers.dev/.well-known/agent-card.json",
    "openapiUrl": null,
    "protocolStd": "a2a",
    "visibility": "public",
    "tags": [],
    "isVerified": false,
    "verification_level": "unclaimed",
    "score": 0.56,
    "lastCheckStatus": "changed",
    "consecutiveFailures": 0,
    "suggestionCount": 1076,
    "lastSuggestedAt": 1790829415,
    "createdAt": 1786986837,
    "ratingAvg": 0,
    "ratingCount": 0,
    "payment": null,
    "isTopMatch": true,
    "_derivedDomains": {
      "manifestDomain": "dac-public-

## 2. Validate and compare the top results

The validator fetches each hosted Agent Card. Keep `VALIDATE_TOP` small while exploring.

### Meaning of the results

| Field | Meaning |
|---|---|
| `relevanceScore` | How closely the registry search matched the agent to `QUERY`; it is not a trust score. |
| `verificationLevel` / `verificationRank` | The registry's identity-verification level and its ordinal rank from 0 to 3. |
| `cardValid` | Whether the validator returned zero error findings for the fetched Agent Card. |
| `readinessScore` | A 0–100 assessment of Agent Card schema, network accessibility, discovery metadata, and trust artifacts. |
| `readinessGrade` | `A+` for 95–100, `A` for 85–94, `B` for 70–84, `C` for 50–69, otherwise `Needs Work`. |
| `errors` / `warnings` | Counts of validator findings that explain deductions and validity. |
| `signatureFindings` | JWS-related findings; a signature is an integrity artifact, not proof that the agent behaves safely. |

### How `readinessScore` is calculated

The registry validator starts at 100, subtracts 25 for each Tier 1 schema error, 15 for each Tier 2 network error, and 5 for each warning. A passing Tier 4 trust check adds 5. Any Tier 1 error caps the result at 40, any Tier 2 error caps it at 20, and the final score is clamped to 0–100. Informational findings do not change the score. See the [official validator implementation](https://github.com/A2ARegistry/GlobalA2ARegistry/blob/main/packages/a2a-validate/src/validator.js#L599-L650).

> **Interpretation:** readiness measures whether the published Agent Card is well formed and discoverable. It does not verify capability accuracy, behavioral reliability, authorization safety, or overall trustworthiness. A high readiness score does not override a weak identity level.

In [6]:
if not agents:
    raise RuntimeError("Search returned no agents. Try a broader QUERY.")

assessments = [assess_agent(agent) for agent in agents[:VALIDATE_TOP]]
print_table(
    assessments,
    ["package", "relevanceScore", "verificationLevel", "verificationRank",
     "verificationMaxRank", "cardValid", "readinessScore",
     "readinessGrade", "errors", "warnings", "signatureFindings"],
    limit=VALIDATE_TOP,
)
print("\nTrust score:", assessments[0]["trustScore"])
print(assessments[0]["trustExplanation"])

package                                            | relevanceScore | verificationLevel | verificationRank | verificationMaxRank | cardValid | readinessScore | readinessGrade | errors | warnings | signatureFindings             
---------------------------------------------------+----------------+-------------------+------------------+---------------------+-----------+----------------+----------------+--------+----------+-------------------------------
dev.workers.digital_agent_cafe                     | 0.56           | unclaimed         | 0                | 3                   | False     | 40             | Needs Work     | 2      | 2        | ['SIGNATURES_OPTIONAL_ABSENT']
com.digitalagentcafe.where_digital_entities_meet_… | 0.51           | unclaimed         | 0                | 3                   | True      | 90             | A              | 0      | 2        | ['SIGNATURES_OPTIONAL_ABSENT']

Trust score: None
Not computed: registry identity and card readiness do not measure beh

In [8]:
print(json.dumps(assessments[0], indent=2, ensure_ascii=False))

{
  "package": "dev.workers.digital_agent_cafe",
  "name": "Digital Agent Cafe",
  "relevanceScore": 0.56,
  "verificationLevel": "unclaimed",
  "verificationRank": 0,
  "verificationMaxRank": 3,
  "cardValid": false,
  "readinessScore": 40,
  "readinessGrade": "Needs Work",
  "errors": 2,
  "warnings": 2,
  "signatureFindings": [
    "SIGNATURES_OPTIONAL_ABSENT"
  ],
  "trustScore": null,
  "trustExplanation": "Not computed: registry identity and card readiness do not measure behavioral trust.",
  "card": {
    "name": "Digital Agent Cafe",
    "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled digital persons. CHAT is available. No monetary payment is required.",
    "supportedInterfaces": [
      {
        "url": "https://dac-public-free-gateway.donuk-kitap.workers.dev",
        "protocolBinding": "HTTP+JSON",
        "protocolVersion": "1.0"
      }
    ],
    "version": "1.0.0",
    "documentationUrl": "https://dac-public-free-auth.donuk-kitap.wo

## 3. Inspect one agent's details

Change `SELECTED_INDEX` to inspect another result from the validated set. The normalized overview is followed by the raw registry record.

In [9]:
SELECTED_INDEX = 0
if not 0 <= SELECTED_INDEX < len(assessments):
    raise IndexError(f"SELECTED_INDEX must be between 0 and {len(assessments) - 1}.")
selected_agent = agents[SELECTED_INDEX]
selected_assessment = assessments[SELECTED_INDEX]

overview = {
    key: selected_agent.get(key)
    for key in [
        "package", "name", "description", "category", "targetAudience",
        "protocol", "manifest", "payment", "lastCheckStatus",
        "consecutiveFailures", "ratingAverage", "ratingCount",
    ]
}
overview.update({
    "relevanceScore": selected_assessment["relevanceScore"],
    "verificationLevel": selected_assessment["verificationLevel"],
    "verificationRank": (
        f"{selected_assessment['verificationRank']}/"
        f"{selected_assessment['verificationMaxRank']}"
    ),
    "cardValid": selected_assessment["cardValid"],
    "readinessScore": selected_assessment["readinessScore"],
    "readinessGrade": selected_assessment["readinessGrade"],
})
print(json.dumps(overview, indent=2, ensure_ascii=False))

print("\nRaw registry record:")
print(json.dumps(selected_agent["rawRegistryRecord"], indent=2, ensure_ascii=False))

{
  "package": "dev.workers.digital_agent_cafe",
  "name": "Digital Agent Cafe",
  "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled digital persons. CHAT is available. No monetary payment is required.",
  "category": "Utilities",
  "targetAudience": "General",
  "protocol": "a2a",
  "manifest": "https://dac-public-free-discovery.donuk-kitap.workers.dev/.well-known/agent-card.json",
  "payment": null,
  "lastCheckStatus": "changed",
  "consecutiveFailures": 0,
  "ratingAverage": 0,
  "ratingCount": 0,
  "relevanceScore": 0.56,
  "verificationLevel": "unclaimed",
  "verificationRank": "0/3",
  "cardValid": false,
  "readinessScore": 40,
  "readinessGrade": "Needs Work"
}

Raw registry record:
{
  "id": "0c1c5117-e8c9-4c3f-bb48-d35a58d7b3dd",
  "orgId": "org_system_unclaimed",
  "packageName": "dev.workers.digital_agent_cafe",
  "displayName": "Digital Agent Cafe",
  "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled 

## 4. Inspect the Agent Card and verification findings

Errors and warnings explain the readiness score. Signature findings show whether the validator found a trust artifact; they do not independently prove the operator's identity or the agent's behavior.

In [10]:
print_table(
    selected_assessment["findings"],
    ["tier", "severity", "code", "title", "message"],
    limit=100,
)

print("\nFetched Agent Card:")
print(json.dumps(selected_assessment["card"], indent=2, ensure_ascii=False))

tier            | severity | code                        | title                                    | message                                           
----------------+----------+-----------------------------+------------------------------------------+---------------------------------------------------
tier2_network   | pass     | HTTP_200_OK                 | HTTP Status 200 OK                       | Manifest endpoint successfully responded with HTT…
tier2_network   | pass     | HTTPS_ENFORCED              | HTTPS Encryption Valid                   | Endpoint is served over secure TLS/HTTPS.         
tier2_network   | pass     | MIME_TYPE_A2A_CANONICAL     | Canonical MIME Content-Type              | Content-Type 'application/a2a+json; charset=utf-8…
tier2_network   | warning  | CORS_MISSING                | Missing CORS Headers                     | No 'Access-Control-Allow-Origin: *' header found.…
tier1_schema    | pass     | PAYLOAD_SIZE_OK             | Payload Size OK        

## What this notebook can and cannot conclude

Use the results to answer whether an agent matched the query, what it claims to do, how strongly the registry verified its identity, and how ready its Agent Card is. For consequential use, separately test capability accuracy, availability, authentication enforcement, permission boundaries, and behavior. Those observations are required before producing a defensible broader trust assessment.